# Customer Segmentation using RFM and K-Means Clustering

In this notebook, we focus on identifying customer groups using two powerful segmentation methodologies:
1. **RFM (Recency, Frequency, Monetary) Analysis**: A heuristic marketing model used to identify a company's best customers by measuring how recently, how often, and how much they spend.
2. **K-Means Clustering**: An unsupervised Machine Learning algorithm used to partition customer profiles into distinct groups based on behavioral patterns.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

## 1. Load Segmented Customers Data
Let's load the dataset resulting from `src/segmentation.py`.

In [ ]:
processed_dir = "../data/processed"
df = pd.read_csv(os.path.join(processed_dir, "segmented_customers.csv"))

print(f"Segmented Dataset Shape: {df.shape}")
df.head()

## 2. RFM Analysis Visualizations
Let's inspect how our RFM marketing segments are distributed.

In [ ]:
# RFM Segment distribution
segment_counts = df['rfm_segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Customer Count']

plt.figure(figsize=(12, 6))
sns.barplot(x='Customer Count', y='Segment', data=segment_counts, palette='viridis')
plt.title('Distribution of Customer RFM Marketing Segments')
plt.xlabel('Number of Customers')
plt.ylabel('Marketing Segment')
plt.tight_layout()
plt.show()

In [ ]:
# Average Spend and Recency per RFM Segment
rfm_metrics = df.groupby('rfm_segment').agg(
    avg_spend=('total_spend', 'mean'),
    avg_recency=('recency_days', 'mean'),
    customer_count=('customer_id', 'count')
).reset_index().sort_values(by='avg_spend', ascending=False)

rfm_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Avg Spend per segment
sns.barplot(x='avg_spend', y='rfm_segment', data=rfm_metrics, ax=axes[0], palette='magma')
axes[0].set_title('Average Customer Lifetime Spend by RFM Segment')
axes[0].set_xlabel('Average Spend ($)')
axes[0].set_ylabel('')

# Avg Recency per segment
sns.barplot(x='avg_recency', y='rfm_segment', data=rfm_metrics, ax=axes[1], palette='plasma')
axes[1].set_title('Average Days Since Last Purchase by RFM Segment')
axes[1].set_xlabel('Average Recency (Days)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 3. Machine Learning K-Means Clustering
Let's review the clustering metrics and profile the generated clusters. 
K-Means was executed on three standardized, log-transformed features: `recency_days`, `total_transactions`, and `total_spend`.

In [ ]:
# Filter out active customers for plotting features
active_df = df[df['cluster_name'] != 'Never Purchased'].copy()

# Plot cluster distribution
cluster_counts = df['cluster_name'].value_counts().reset_index()
cluster_counts.columns = ['Cluster', 'Count']

plt.figure(figsize=(10, 5))
sns.barplot(x='Count', y='Cluster', data=cluster_counts, palette='Set1')
plt.title('K-Means Cluster Size Distribution')
plt.xlabel('Number of Customers')
plt.ylabel('K-Means Cluster')
plt.show()

### Visualizing Clusters (Spend vs Recency vs Frequency)

In [ ]:
# 2D projection of Spend vs Recency
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='recency_days', 
    y='total_spend', 
    hue='cluster_name', 
    size='total_transactions', 
    sizes=(20, 200),
    data=active_df, 
    palette='Set1',
    alpha=0.7
)
plt.title('K-Means Clusters: Lifetime Spend vs Recency')
plt.xlabel('Recency (Days Since Last Purchase)')
plt.ylabel('Total Lifetime Spend ($)')
plt.yscale('log') # Log scale because of skewness
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Cluster Profile Analysis
Let's check the mean behavioral and demographic properties for each cluster to understand who they are.

In [ ]:
# Profile clusters
profile = df.groupby('cluster_name').agg(
    customer_count=('customer_id', 'count'),
    avg_spend=('total_spend', 'mean'),
    avg_transactions=('total_transactions', 'mean'),
    avg_recency=('recency_days', 'mean'),
    avg_age=('age', 'mean'),
    avg_income=('annual_income', 'mean'),
    avg_satisfaction=('satisfaction_score', 'mean')
).reset_index().sort_values(by='avg_spend', ascending=False)

profile

## 5. Strategic Business Recommendations
Based on the cluster profiling, we formulate targeted marketing and customer engagement campaigns:

1. **VIP Spenders (High Spend, High Frequency, High Recency)**
   - **Characteristics**: The absolute core of the business. Very high spend and transaction frequency. Highly satisfied.
   - **Strategy**: Reward loyalty. Set up a premium reward tier, provide early access to new releases, invite to exclusive VIP programs, and request referrals.

2. **Active Customers (Moderate Spend, Frequent, Low Recency)**
   - **Characteristics**: Regular buyers who shop consistently but have slightly lower average spend than VIPs.
   - **Strategy**: Cross-sell and up-sell. Provide product bundles based on their preferred categories to increase Average Order Value (AOV). Set up subscription options for consumable products.

3. **Slipping Customers (Moderate Spend, Lower Frequency, High Recency)**
   - **Characteristics**: These customers bought multiple times but haven't purchased in a long time (high recency).
   - **Strategy**: Re-engagement campaigns. Send personalized win-back emails, offer special discounts or coupons, and showcase new arrivals in their preferred category.

4. **Lost Customers (Low Spend, Very Low Frequency, Very High Recency)**
   - **Characteristics**: Customers who bought only once or twice a long time ago. Satisfaction is often lower.
   - **Strategy**: Reactivate or ignore. Low priority. Send a survey to ask why they left, or run a high-discount email blast to see if any can be reactivated cheaply.